# AI-Powered Resume Analyzer

A beginner-friendly Python project that compares a resume with a job description, calculates a text similarity score, identifies matching/missing skills, and displays a simple chart.

**How to use:** Run the cells from top to bottom. To analyze your own resume, put a PDF named `resume.pdf` in the same folder as this notebook. If no PDF is found, the notebook uses built-in sample resume text.

## 1. Import libraries

In [ ]:
import os
import PyPDF2
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

## 2. Add a sample resume
This sample lets you test the project immediately, even before adding a PDF.

In [ ]:
sample_resume = '''
Name: Rahul Sharma
Education: B.Tech in Computer Science
Skills: Python, SQL, Excel, Power BI, Machine Learning, Data Analysis,
Statistics, Pandas, NumPy, Data Visualization
Experience: Completed a Data Analyst internship. Worked on sales analysis
and data visualization projects.
Projects: Sales Data Analysis using Python; Customer Segmentation;
Interactive Dashboard using Power BI.
'''

resume_text = sample_resume
print(resume_text)

## 3. Add the job description
Replace this text with any job description you want to compare against.

In [ ]:
job_description = '''
We are looking for a Data Analyst.
Required skills: Python, SQL, Excel, Power BI, Tableau, Machine Learning,
Statistics, Data Visualization, Pandas, NumPy, Communication Skills.
Candidates should understand data analysis, data cleaning, dashboard
development and reporting. A degree in Computer Science or Data Science
is preferred.
'''

print("Job description loaded.")
print(job_description)

## 4. Extract text from a PDF resume

Put your PDF in the same folder as this notebook and name it `resume.pdf`. If the file is not present, the sample resume remains active.

In [ ]:
def extract_resume_text(pdf_path):
    extracted_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                extracted_text += page_text + "\n"
    return extracted_text

resume_path = "resume.pdf"  # Change this if your PDF has a different name or location

if os.path.exists(resume_path):
    pdf_resume_text = extract_resume_text(resume_path)
    if pdf_resume_text.strip():
        resume_text = pdf_resume_text
        print("PDF resume loaded successfully!")
        print(resume_text[:1500])
    else:
        print("No readable text found in the PDF. Using the sample resume instead.")
else:
    print("resume.pdf was not found. Using the sample resume for now.")
    print("Add resume.pdf beside this notebook to analyze your own resume.")

## 5. Calculate the resume-job match score

TF-IDF converts text into numerical features. Cosine similarity compares the two text vectors. The score is a text-similarity measure, not a hiring probability.

In [ ]:
def calculate_match(resume, job_description):
    documents = [resume, job_description]
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(documents)
    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return round(similarity * 100, 2)

match_score = calculate_match(resume_text, job_description)
print(f"Resume Match Score: {match_score}%")

if match_score >= 75:
    match_category = "Excellent text match"
elif match_score >= 50:
    match_category = "Good text match"
elif match_score >= 30:
    match_category = "Average text match"
else:
    match_category = "Low text match"

print("Match Category:", match_category)

## 6. Identify matching and missing skills

This uses a simple editable skills list and keyword matching. Add more skills to the list as needed.

In [ ]:
skills_list = [
    "python", "sql", "excel", "power bi", "tableau",
    "machine learning", "statistics", "data analysis",
    "data visualization", "pandas", "numpy",
    "communication skills", "data cleaning", "reporting"
]

resume_lower = resume_text.lower()
job_lower = job_description.lower()

required_skills = [skill for skill in skills_list if skill in job_lower]
matched_skills = [skill for skill in required_skills if skill in resume_lower]
missing_skills = [skill for skill in required_skills if skill not in resume_lower]

print("Required skills:", required_skills)
print("\nMatched skills:", matched_skills)
print("\nMissing skills:", missing_skills)

## 7. Display a skill analysis table

In [ ]:
skill_results = pd.DataFrame({
    "Skill": required_skills,
    "Status": ["Matched" if skill in matched_skills else "Missing" for skill in required_skills]
})

display(skill_results)

## 8. Visualize the results

In [ ]:
matched_count = len(matched_skills)
missing_count = len(missing_skills)

plt.figure(figsize=(7, 4))
plt.bar(["Matched Skills", "Missing Skills"], [matched_count, missing_count])
plt.title("Resume Skill Analysis")
plt.ylabel("Number of Skills")
plt.show()

plt.figure(figsize=(5, 5))
plt.pie(
    [matched_count, missing_count],
    labels=["Matched", "Missing"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Required Skills Coverage")
plt.show()

## 9. Generate improvement suggestions

In [ ]:
print("RESUME ANALYSIS SUMMARY")
print("-" * 35)
print(f"Match score: {match_score}%")
print(f"Match category: {match_category}")
print(f"Matched skills: {len(matched_skills)}")
print(f"Missing skills: {len(missing_skills)}")

if missing_skills:
    print("\nSuggestions:")
    for skill in missing_skills:
        print(f"- If you genuinely have this skill, add relevant evidence of it to your resume: {skill.title()}")

if not missing_skills:
    print("\nAll skills in this simple checklist were found in the resume.")

print("\nTip: Tailor your project descriptions and experience to the job requirements. Never add skills you do not have.")

## 10. Conclusion

This notebook demonstrates PDF text extraction, TF-IDF text representation, cosine similarity, keyword-based skill checking, and basic visualizations.

**Limitations:** The match score is based on text similarity and does not assess a candidate's true ability. Keyword matching can miss synonyms and context. Scanned PDFs may require OCR, which is not included in this version.